In [ ]:
! pip3 install -U accelerate
! pip3 install datasets --upgrade
! pip3 install evaluate --upgrade
! pip3 install transformers --upgrade
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import torch


  Using cached accelerate-1.6.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.6.0-py3-none-any.whl (354 kB)
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.26.0
    Uninstalling accelerate-0.26.0:
      Successfully uninstalled accelerate-0.26.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 10.5 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 13.0.0
    Uninstalling pyarrow-13.0.0:
      Successfully uninstalled pyarrow-13.0.0
  Attempting uninstall: datasets
    Found existing installation: datasets 2.19.1
    Uninstalling datasets-2.19.1:
      Successfully uninstalled datasets-2.19.1


/opt/homebrew/Caskroom/miniconda/base/envs/myenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/homebrew/Caskroom/miniconda/base/envs/myenv/lib/python3.11/site-packages/torchvision/io/image.py:14: UserWarning: Failed to load image Python extension: 'dlopen(/opt/homebrew/Caskroom/miniconda/base/envs/myenv/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libjpeg.9.dylib
  Referenced from: <EB3FF92A-5EB1-3EE8-AF8B-5923C1265422> /opt/homebrew/Caskroom/miniconda/base/envs/myenv/lib/python3.11/site-packages/torchvision/image.so
  Reason: tried: '/opt/homebrew/Caskroom/miniconda/base/envs/myenv/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/opt/homebrew/Caskroom/miniconda/base/envs/myenv/lib/python3.11/site-packages/torchvision/.

In [ ]:
def remap_labels(example):
    example["label"] = 0 if example["target"] == 0 else 1
    return example

dataset = load_dataset("dffesalbon/dota-2-toxic-chat-data")
dataset = dataset.map(remap_labels)

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


In [3]:
def preprocess_function(examples):
    return tokenizer(examples["message"], truncation=True, padding="max_length", max_length=128)

# Tokenize the dataset
tokenized_datasets = dataset.map(preprocess_function, batched=True)

# Set format for PyTorch tensors
tokenized_datasets = tokenized_datasets.remove_columns(["message", "target"])
tokenized_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


In [ ]:
!pip show transformers
train_dataset = tokenized_datasets["train"]
test_dataset = tokenized_datasets["test"]

model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",         # Evaluate every epoch (older argument name)
    save_strategy="epoch",         # Save model every epoch
    learning_rate=2e-5,            # Learning rate
    per_device_train_batch_size=16,  # Batch size for training
    per_device_eval_batch_size=16,   # Batch size for evaluation
    num_train_epochs=3,            # Number of training epochs
    weight_decay=0.01,             # Weight decay
    logging_dir="./logs",          # Directory for storing logs
    logging_steps=10,              # Log every 10 steps
    save_total_limit=2,            # Limit the number of saved checkpoints
    load_best_model_at_end=True,   # Load the best model at the end of training
    metric_for_best_model="accuracy"  # Metric to determine the best model
)

Name: transformers
Version: 4.51.1
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /opt/homebrew/Caskroom/miniconda/base/envs/myenv/lib/python3.11/site-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: 


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,  # Use data_collator instead of tokenizer
    compute_metrics=compute_metrics
)
trainer.train()

trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.364000,0.402651,0.836991,0.827815,0.783699,0.877193
2,0.153300,0.333165,0.876176,0.862609,0.855172,0.870175
3,0.109500,0.353213,0.885580,0.874355,0.858108,0.891228


{'eval_loss': 0.3532129228115082,
 'eval_accuracy': 0.8855799373040752,
 'eval_f1': 0.8743545611015491,
 'eval_precision': 0.8581081081081081,
 'eval_recall': 0.8912280701754386,
 'eval_runtime': 15.1846,
 'eval_samples_per_second': 42.016,
 'eval_steps_per_second': 2.634,
 'epoch': 3.0}